In [0]:
from pyspark.sql.functions import DataFrame
from pyspark.sql import functions as F

In [0]:
df_billings: DataFrame = spark.table("post_renewal_churn.raw.billings")
df_billings.describe().display()

In [0]:
df_billings = df_billings\
.filter(F.col("prospect_outcome") != "Open")\
    .withColumn("datediff", F.datediff("closed_date", "prospect_renewal_date"))\
        .filter(F.col("datediff") < 29)

# add an index column to represent each row uniquely
df_billings = df_billings.withColumn("index", F.monotonically_increasing_id())

In [0]:
df_billings.display()

In [0]:
f_billings = df_billings.select(
    "index",
    "co_ref",
    "prospect_renewal_date",
    "closed_date",
    "datediff",
    "total_renewal_score_new",
    "status_scores",
    "sustainability_score",
    "auto_renewal_score",
    "renewal_score_at_release",
    "anchoring_score",
    "current_anchorings",
    "tenure_years",
    "last_years_price",
    "prospect_outcome"
)
f_billings.display()

In [0]:
dfr=spark.table("post_renewal_churn.raw.renewal_calls").toPandas()
dfr[dfr["Analysed_Call"]!=1].notnull().sum()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── 1. Load / prepare billings ──────────────────────────────────────────
df_billings = spark.table("post_renewal_churn.raw.billings") \
    .filter(F.col("prospect_outcome") != "Open") \
    .withColumn("datediff", F.datediff("closed_date", "prospect_renewal_date")) \
    .filter(F.col("datediff") < 29) \
    .withColumn("index", F.monotonically_increasing_id())

# Select only needed billing columns
df_billings = df_billings.select(
    "index",
    "co_ref",
    "prospect_renewal_date",
    "closed_date",
    "datediff",
    "total_renewal_score_new",
    "status_scores",
    "sustainability_score",
    "auto_renewal_score",
    "renewal_score_at_release",
    "anchoring_score",
    "current_anchorings",
    "tenure_years",
    "last_years_price",
    "prospect_outcome"
)

# ── 2. Load / prepare renewal_calls ─────────────────────────────────────
call_cols = [
    "co_ref",
    "call_date",
    "call_direction",
    "customer_reaction_category",
    "agent_renewal_pitch_category",
    "customer_renewal_response_category",
    "membership_renewal_decision",
    "serious_complaint",
    "other_complaint",
    "discussion_on_price_increase",
    "renewal_impact_due_to_price_increase",
    "discount_or_waiver_requested",
    "discount_offered",
    "explicit_competitor_mention",
    "explicit_switching_intent",
    "desire_to_cancel",
    "customer_response",
    "agent_renewal_initiation",
    "call_number"
]

df_calls = spark.table("post_renewal_churn.raw.renewal_calls").select(*call_cols + ["analysed_call"]) \
    .filter(F.col("analysed_call") == 1) \
    .drop("analysed_call").withColumn("call_date", F.to_date("call_date"))

df_calls = df_calls.dropna(subset=["customer_renewal_response_category"])
df_calls = df_calls.filter(F.col("customer_renewal_response_category") != "Not Mentioned")
#df_calls=df_calls.filter(")
df_calls = spark.table("post_renewal_churn.raw.renewal_calls").select(*call_cols) \
    .withColumn("call_date", F.to_date("call_date"))

# ── 3. Join on co_ref, keeping only calls AFTER prospect_renewal_date ───
df_billings = df_billings \
    .withColumn("prospect_renewal_date", F.to_date("prospect_renewal_date"))

df_joined = df_billings.join(
    df_calls,
    on="co_ref",
    how="left"
).filter(
    F.col("call_date") > F.col("prospect_renewal_date")   # only post-renewal calls
)

# ── 4. Keep only the LATEST call per billing row (index) ────────────────
window = Window.partitionBy("index").orderBy(F.desc("call_date"))

df_latest_call = df_joined \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

df_latest_call.display()

In [0]:
df_latest_call.toPandas().notnull().sum()

In [0]:
df_latest_call.toPandas().notnull().sum()

In [0]:
email=spark.table("post_renewal_churn.raw.emails").toPandas()
email.notnull().sum()


In [0]:
# emails dataset — select useful columns
email_cols = [
    "co_ref", "time_to_renewal", "year",
    "crm_contractor_sentiment", "crm_contractor_sentiment_score",
    "crm_customer_payment_intention", "crm_competitors_mentioned",
    "crm_dissatisified_with_renewal_price", "crm_customer_complained",
    "crm_negative_customer_experience", "crm_dissatisfaction_with_support",
    "crm_financial_hardship_mentioned", "crm_refund_mentioned",
    "crm_contractor_suggested_leave", "crm_membership_overdue",
    "crm_agent_chase_count", "crm_contractor_engagement"
]

df_emails = spark.table("post_renewal_churn.raw.emails").select(*email_cols)

# Join to your existing df_latest_call (billings + calls)
# Match on co_ref and year (from Renewal_Year in billings)
df_final = df_latest_call \
    .withColumn("renewal_year", F.year("prospect_renewal_date")) \
    .join(
        df_emails.withColumnRenamed("year", "email_year"),
        on=[
            F.col("co_ref") == F.col("co_ref"),
            F.col("renewal_year") == F.col("email_year")
        ],
        how="left"
    ).drop("email_year")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── 1. Load / prepare billings ──────────────────────────────────────────
df_billings = spark.table("`post_renewal_churn`.`raw`.`billings`") \
    .filter(F.col("prospect_outcome") != "Open") \
    .withColumn("datediff", F.datediff("closed_date", "prospect_renewal_date")) \
    .filter(F.col("datediff") < 29) \
    .withColumn("index", F.monotonically_increasing_id())

df_billings = df_billings.select(
    "index",
    "co_ref",
    "prospect_renewal_date",
    "closed_date",
    "datediff",
    "total_renewal_score_new",
    "status_scores",
    "sustainability_score",
    "auto_renewal_score",
    "renewal_score_at_release",
    "anchoring_score",
    "current_anchorings",
    "tenure_years",
    "last_years_price",
    "prospect_outcome"
).withColumn("prospect_renewal_date", F.to_date("prospect_renewal_date")) \
 .withColumn("renewal_year", F.year("prospect_renewal_date"))

# ── 2. Load / prepare renewal_calls ─────────────────────────────────────
call_cols = [
    "co_ref",
    "call_date",
    "call_direction",
    "customer_reaction_category",
    "agent_renewal_pitch_category",
    "customer_renewal_response_category",
    "membership_renewal_decision",
    "serious_complaint",
    "other_complaint",
    "discussion_on_price_increase",
    "renewal_impact_due_to_price_increase",
    "discount_or_waiver_requested",
    "discount_offered",
    "explicit_competitor_mention",
    "explicit_switching_intent",
    "desire_to_cancel",
    "customer_response",
    "agent_renewal_initiation",
    "call_number",
    "analysed_call"
]

df_calls = spark.table("`post_renewal_churn`.`raw`.`renewal_calls`").select(*call_cols) \
    .filter(F.col("analysed_call") == 1) \
    .drop("analysed_call") \
    .withColumn("call_date", F.to_date("call_date"))
df_calls = df_calls.dropna(subset=["customer_renewal_response_category"])
df_calls = df_calls.filter(F.col("customer_renewal_response_category") != "Not Mentioned")

# ── 3. Load / prepare emails ─────────────────────────────────────────────
email_cols = [
    "co_ref",
    "year",
    "crm_contractor_sentiment",
    "crm_contractor_sentiment_score",
    "crm_customer_payment_intention",
    "crm_competitors_mentioned",
    "crm_dissatisified_with_renewal_price",
    "crm_customer_complained",
    "crm_negative_customer_experience",
    "crm_dissatisfaction_with_support",
    "crm_financial_hardship_mentioned",
    "crm_refund_mentioned",
    "crm_contractor_suggested_leave",
    "crm_membership_overdue",
    "crm_agent_chase_count",
    "crm_contractor_engagement"
]

df_emails = spark.table("`post_renewal_churn`.`raw`.`emails`").select(*email_cols) \
    .withColumnRenamed("year", "email_year")

# ── 4. Join billings with calls ──────────────────────────────────────────
# Keep only calls AFTER prospect_renewal_date, then take the latest one per billing row

df_joined = df_billings.join(
    df_calls,
    on=df_billings["co_ref"] == df_calls["co_ref"],
    how="left"
).filter(
    F.col("call_date") > F.col("prospect_renewal_date")
).drop(df_calls["co_ref"])

# Keep only the latest call per billing row
window_calls = Window.partitionBy("index").orderBy(F.desc("call_date"))

df_latest_call = df_joined \
    .withColumn("rn", F.row_number().over(window_calls)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

# ── 5. Join with emails ───────────────────────────────────────────────────
# Match on co_ref + renewal year so each billing row gets its relevant email features

df_final = df_latest_call.join(
    df_emails,
    on=(df_latest_call["co_ref"] == df_emails["co_ref"]) &
       (df_latest_call["renewal_year"] == df_emails["email_year"]),
    how="left"
).drop(df_emails["co_ref"]) \
 .drop("email_year")

# ── 6. Display result ─────────────────────────────────────────────────────
print("Final shape:", (df_final.count(), len(df_final.columns)))
print("Columns:", df_final.columns)
df_final.display()

In [0]:
df_final.toPandas().notnull().sum()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── 1. Load / prepare billings ──────────────────────────────────────────
df_billings = spark.table("workspace.`ds-raw-datasets`.raw_billings") \
    .filter(F.col("prospect_outcome") != "Open") \
    .withColumn("datediff", F.datediff("closed_date", "prospect_renewal_date")) \
    .filter(F.col("datediff") < 29) \
    .withColumn("index", F.monotonically_increasing_id())

df_billings = df_billings.select(
    "index",
    "co_ref",
    "prospect_renewal_date",
    "closed_date",
    "datediff",
    "total_renewal_score_new",
    "status_scores",
    "sustainability_score",
    "auto_renewal_score",
    "renewal_score_at_release",
    "anchoring_score",
    "current_anchorings",
    "tenure_years",
    "last_years_price",
    "prospect_outcome"
).withColumn("prospect_renewal_date", F.to_date("prospect_renewal_date")) \
 .withColumn("closed_date", F.to_date("closed_date"))

# ── 2. Load / prepare renewal_calls ─────────────────────────────────────
call_cols = [
    "co_ref",
    "call_date",
    "call_direction",
    "customer_reaction_category",
    "agent_renewal_pitch_category",
    "customer_renewal_response_category",
    "membership_renewal_decision",
    "serious_complaint",
    "other_complaint",
    "discussion_on_price_increase",
    "renewal_impact_due_to_price_increase",
    "discount_or_waiver_requested",
    "discount_offered",
    "explicit_competitor_mention",
    "explicit_switching_intent",
    "desire_to_cancel",
    "customer_response",
    "agent_renewal_initiation",
    "call_number"
]

df_calls = spark.table("workspace.`ds-raw-datasets`.raw_renewal_calls") \
    .select(*call_cols + ["analysed_call"]) \
    .filter(F.col("analysed_call") == "1") \
    .drop("analysed_call") \
    .withColumn("call_date", F.to_date("call_date")) \
    .filter(F.col("customer_renewal_response_category") != "null") \
    .filter(F.col("customer_renewal_response_category") != "Not Mentioned")

# ── 3. Join on co_ref — explicit condition to avoid ambiguous reference ──
df_joined = df_billings.join(
    df_calls,
    on=df_billings["co_ref"] == df_calls["co_ref"],
    how="left"
).filter(
    (F.col("call_date") >= F.col("prospect_renewal_date")) &
    (F.col("call_date") <= F.col("closed_date"))
).drop(df_calls["co_ref"])

# ── 4. Keep only the LATEST call per billing row (index) ────────────────
window = Window.partitionBy("index").orderBy(F.desc("call_date"))

df_latest_call = df_joined \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

df_latest_call.display()